# Classificació C vs I (Oz, 1000-1500 ms)

Flux net i directe:
- Dades EEG preprocessades en format .fif
- Finestra temporal de 1000 a 1500 ms
- Canal Oz
- Classes: Congruent (C) vs Incongruent (I = Pur + Mixt)
- Caracteristiques oscil.latories amb STFT (finestra 500 ms, pas 0.02 s) en banda alpha (8-12 Hz)
- Deteccio d'atipics amb distancia de Mahalanobis (sense eliminar-los)
- Classificacio amb PCA + LDA i validacio creuada

In [1]:
import sys
from pathlib import Path

%pip install -q mne
import mne
import numpy as np
from scipy.signal import stft
from scipy.stats import chi2
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import warnings

# Filtrem avisos coneguts no critics d'MNE
warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
    module="mne",
    message=".*does not conform to MNE naming conventions.*",
)
warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
    module="mne",
    message=".*events passed to the Epochs constructor are not chronologically ordered.*",
)

mne.set_log_level("WARNING")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    possible_paths = [
        Path("/content/drive/Shareddrives/TFG_Nils_2026"),
        Path("/content/drive/MyDrive/TFG_Nils_2026"),
    ]
    BASE_DIR = next((p for p in possible_paths if p.exists()), possible_paths[0])
    DATA_DIR = BASE_DIR / "02_Data" / "01_Preprocessed_epochs"
else:
    BASE_DIR = Path(".").resolve()
    DATA_DIR = BASE_DIR / "data"

TARGET_CH = "Oz"
TARGET_WINDOW = (1.0, 1.5)
ALPHA_FREQS = np.arange(8, 13, 1)
STFT_WIN_SEC = 0.5
STFT_STEP_SEC = 0.02
COND_EVENTS = {
    "Congruent": ["event_1", "event_2"],
    "Incongruent": ["event_3", "event_4", "event_5", "event_6", "event_7", "event_8"],
    "Incongruent Pur": ["event_3", "event_4", "event_5", "event_6"],
    "Incongruent Mixt": ["event_7", "event_8"]
}

fitxers = sorted(DATA_DIR.glob("*.fif"))
print(f"Entorn: {'Colab' if IN_COLAB else 'Local'}")
print(f"Directori dades: {DATA_DIR}")
print(f"Fitxers trobats: {len(fitxers)}")
print(f"Banda usada per STFT: {ALPHA_FREQS[0]}-{ALPHA_FREQS[-1]} Hz")

Mounted at /content/drive
Entorn: Colab
Directori dades: /content/drive/MyDrive/TFG_Nils_2026/02_Data/01_Preprocessed_epochs
Fitxers trobats: 47
Banda usada per STFT: 8-12 Hz


In [ ]:
X_list, y_list, subj_list = [], [], []

for fif_path in fitxers:
    ep = mne.read_epochs(fif_path, preload=True, verbose=False)
    # Corregir unitats i tipus (surten com a 'misc' del preprocessat)
    for ch in ep.info["chs"]:
        ch["unit"] = mne.io.constants.FIFF.FIFF_UNIT_V
    ep.set_channel_types({ch: "eeg" for ch in ep.ch_names}, on_unit_change="ignore")

    if TARGET_CH not in ep.ch_names:
        print(f"Advertencia: {fif_path.name} no te el canal {TARGET_CH}. S'omet aquest fitxer.")
        continue

    ep = ep.copy().pick([TARGET_CH]).crop(tmin=TARGET_WINDOW[0], tmax=TARGET_WINDOW[1])
    c_names = [n for n in COND_EVENTS["Congruent"] if n in ep.event_id]
    i_names = [n for n in COND_EVENTS["Incongruent"] if n in ep.event_id]

    if not c_names or not i_names:
        continue

    # Evitem concatenate_epochs per no perdre ordre cronologic dels events
    selected_names = c_names + i_names
    ep_bin = ep[selected_names]
    c_codes = [ep.event_id[n] for n in c_names]
    y_subj = np.where(np.isin(ep_bin.events[:, 2], c_codes), 0, 1)

    x_data = ep_bin.get_data().astype(np.float32, copy=False)  # (n_epochs, 1, n_times)
    sfreq = ep_bin.info["sfreq"]
    nperseg = int(round(STFT_WIN_SEC * sfreq))
    hop = int(round(STFT_STEP_SEC * sfreq))
    noverlap = max(0, nperseg - hop)
    nfft = max(nperseg, int(round(sfreq)))

    f, t, Z = stft(
        x_data,
        fs=sfreq,
        window="hann",
        nperseg=nperseg,
        noverlap=noverlap,
        nfft=nfft,
        boundary="zeros",
        padded=True,
        axis=-1,
    )
    power = np.abs(Z) ** 2  # (n_epochs, 1, n_freq, n_t)

    freq_idx = np.array([np.argmin(np.abs(f - fr)) for fr in ALPHA_FREQS])
    power_sel = power[:, :, freq_idx, :]
    X_subj = power_sel.reshape(power_sel.shape[0], -1)

    X_list.append(X_subj)
    y_list.append(y_subj)
    subj_list.extend([fif_path.stem] * len(y_subj))

if not X_list:
    raise RuntimeError("No hi ha dades suficients per fer C vs I amb Oz a 1000-1500 ms.")

X = np.vstack(X_list)
y = np.concatenate(y_list)
subj_list = np.array(subj_list)

print(f"Mostres totals: {len(y)}")
print(f"Congruent (0): {(y == 0).sum()} | Incongruent (1): {(y == 1).sum()}")
print(f"Subjectes inclosos: {len(np.unique(subj_list))}")
print(f"Matriu de caracteristiques: {X.shape}")

Mostres totals: 22860
Congruent (0): 8471 | Incongruent (1): 14389
Subjectes inclosos: 47
Matriu de caracteristiques: (22860, 135)
Freqs STFT usades (alpha): [ 8  9 10 11 12]


In [3]:
def mahalanobis_outliers(data):
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)

    pca_m = PCA(n_components=0.95, random_state=42)
    data_pca = pca_m.fit_transform(data_scaled)

    mean = np.mean(data_pca, axis=0)
    cov = np.cov(data_pca, rowvar=False)
    inv_cov = np.linalg.pinv(cov)

    diff = data_pca - mean
    md2 = np.einsum("ij,jk,ik->i", diff, inv_cov, diff)
    md = np.sqrt(np.clip(md2, 0, None))

    threshold = np.sqrt(chi2.ppf(0.999, df=data_pca.shape[1]))
    return md, md > threshold


md_values, is_outlier = mahalanobis_outliers(X)
print(f"Atipics detectats: {is_outlier.sum()} de {len(X)}")
print("Nota: no els eliminem en aquesta versió.")

Atipics detectats: 839 de 22860
Nota: no els eliminem en aquesta versió.


In [4]:
clf_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=0.90, random_state=42)),
        ("lda", LDA(priors=[0.5, 0.5])),
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(clf_pipeline, X, y, cv=cv, scoring="accuracy")

print("Resultats C vs I (Accuracy):")
print(f"Mitjana: {scores.mean():.3f}")
print(f"Desviacio: {scores.std():.3f}")
print(f"Scores per fold: {np.round(scores, 3)}")

# Report detallat per un unic split: ultim fold
splits = list(cv.split(X, y))
train_idx, test_idx = splits[-1]
clf_last = clone(clf_pipeline)
clf_last.fit(X[train_idx], y[train_idx])
y_pred_last = clf_last.predict(X[test_idx])

print("\nDetall ultim fold (per comprovar biaix de classe):")
print("Confusion matrix [files=true, columnes=pred]:")
print(confusion_matrix(y[test_idx], y_pred_last))
print("Classification report:")
print(
    classification_report(
        y[test_idx],
        y_pred_last,
        target_names=["Congruent (C)", "Incongruent (I)"],
        digits=3,
    )
)

Resultats C vs I (Accuracy):
Mitjana: 0.598
Desviacio: 0.008
Scores per fold: [0.584 0.599 0.607 0.601 0.6  ]

Detall ultim fold (per comprovar biaix de classe):
Confusion matrix [files=true, columnes=pred]:
[[ 497 1197]
 [ 633 2245]]
Classification report:
                 precision    recall  f1-score   support

  Congruent (C)      0.440     0.293     0.352      1694
Incongruent (I)      0.652     0.780     0.710      2878

       accuracy                          0.600      4572
      macro avg      0.546     0.537     0.531      4572
   weighted avg      0.574     0.600     0.578      4572

